# Kidney outcomes analysis

Original MED264 analysis, prepared for version control.

Run only in an authorized All of Us Researcher Workbench with Registered Tier v7 access. Outputs and notebook metadata have been cleared for distribution. Review `docs/analysis-notes.md` for fixes and unresolved methodological limitations. The corrected analysis has not been rerun on participant data. Never commit executed participant-data outputs.


In [ ]:
import os
import pandas as pd
import numpy as np

if not os.environ.get("WORKSPACE_CDR"):
    raise RuntimeError("Run this notebook in an authorized All of Us Researcher Workbench.")


In [ ]:
import pandas
import os

# This query represents dataset "dataset_with_diseases" for domain "person" and was generated for All of Us Registered Tier Dataset v7
dataset_33371088_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    (concept_id IN(SELECT
                        DISTINCT c.concept_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id       
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                        WHERE
                            concept_id IN (201820, 201826)       
                            AND full_text LIKE '%_rank1]%'      ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) 
                    AND is_standard = 1 ) 
                    AND age_at_event >= 18 ) criteria 
            UNION
            DISTINCT SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                WHERE
                    person_id IN (SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                    WHERE
                        concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                            WHERE
                                concept_id IN (442793, 443732, 4193704)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1) 
                        AND is_standard = 1 
                    UNION
                    DISTINCT SELECT
                        person_id 
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                    WHERE
                        concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                            WHERE
                                concept_id IN (1567956, 44833365)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 0 
                            AND is_selectable = 1) 
                        AND is_standard = 0 ) 
                    AND age_at_event >= 18 ) criteria ) 
                    AND cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (43526465)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (44785829)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (45774751)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (793293)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (45774435)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (1583722)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (40170911)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (44506754)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT ca.descendant_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                            JOIN
                                (SELECT
                                    DISTINCT c.concept_id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id             
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                    WHERE
                                        concept_id IN (793143)             
                                        AND full_text LIKE '%_rank1]%'       ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) b 
                                    ON (ca.ancestor_id = b.concept_id)) 
                                AND is_standard = 1)) criteria ) 
                        AND cb_search_person.person_id IN (SELECT
                            criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN (46236952, 3053283, 40764999, 40771922) 
                            AND is_standard = 1 )) criteria ) )"""

dataset_33371088_person_df = pandas.read_gbq(
    dataset_33371088_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_33371088_person_df.head(5)

In [ ]:
import pandas
import os

# This query represents dataset "dataset_with_diseases" for domain "drug" and was generated for All of Us Registered Tier Dataset v7
dataset_33371088_drug_sql = """
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_standard_concept.vocabulary_id as standard_vocabulary,
        d_exposure.drug_exposure_start_datetime,
        d_exposure.drug_exposure_end_datetime,
        d_exposure.verbatim_end_date,
        d_exposure.drug_type_concept_id,
        d_type.concept_name as drug_type_concept_name,
        d_exposure.stop_reason,
        d_exposure.refills,
        d_exposure.quantity,
        d_exposure.days_supply,
        d_exposure.sig,
        d_exposure.route_concept_id,
        d_route.concept_name as route_concept_name,
        d_exposure.lot_number,
        d_exposure.visit_occurrence_id,
        d_visit.concept_name as visit_occurrence_concept_name,
        d_exposure.drug_source_value,
        d_exposure.drug_source_concept_id,
        d_source_concept.concept_name as source_concept_name,
        d_source_concept.concept_code as source_concept_code,
        d_source_concept.vocabulary_id as source_vocabulary,
        d_exposure.route_source_value,
        d_exposure.dose_unit_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.drug_exposure` d_exposure 
        WHERE
            (
                drug_concept_id IN (SELECT
                    DISTINCT ca.descendant_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                JOIN
                    (SELECT
                        DISTINCT c.concept_id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id             
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                        WHERE
                            concept_id IN (1583722, 40170911, 43526465, 44506754, 44785829, 45774435, 45774751, 793143, 793293)             
                            AND full_text LIKE '%_rank1]%'       ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) b 
                        ON (ca.ancestor_id = b.concept_id)))  
                    AND (d_exposure.PERSON_ID IN (SELECT
                        distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (201820, 201826)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 ) 
                            AND age_at_event >= 18 ) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            person_id IN (SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                    WHERE
                                        concept_id IN (442793, 443732, 4193704)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) 
                                AND is_standard = 1 
                            UNION
                            DISTINCT SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                    WHERE
                                        concept_id IN (1567956, 44833365)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 0 
                                    AND is_selectable = 1) 
                                AND is_standard = 0 ) 
                            AND age_at_event >= 18 ) criteria ) 
                            AND cb_search_person.person_id IN (SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (43526465)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (44785829)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (45774751)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (793293)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (45774435)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (1583722)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (40170911)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (44506754)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (793143)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria ) 
                                AND cb_search_person.person_id IN (SELECT
                                    criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN (46236952, 3053283, 40764999, 40771922) 
                                    AND is_standard = 1 )) criteria ) )
                        )
                    ) d_exposure 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_standard_concept 
                        ON d_exposure.drug_concept_id = d_standard_concept.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_type 
                        ON d_exposure.drug_type_concept_id = d_type.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_route 
                        ON d_exposure.route_concept_id = d_route.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                        ON d_exposure.visit_occurrence_id = v.visit_occurrence_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_visit 
                        ON v.visit_concept_id = d_visit.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` d_source_concept 
                        ON d_exposure.drug_source_concept_id = d_source_concept.concept_id"""

dataset_33371088_drug_df = pandas.read_gbq(
    dataset_33371088_drug_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_33371088_drug_df.head(5)

In [ ]:
import pandas
import os

# This query represents dataset "dataset_with_diseases" for domain "measurement" and was generated for All of Us Registered Tier Dataset v7
dataset_33371088_measurement_sql = """
    SELECT
        measurement.person_id,
        measurement.measurement_concept_id,
        m_standard_concept.concept_name as standard_concept_name,
        m_standard_concept.concept_code as standard_concept_code,
        m_standard_concept.vocabulary_id as standard_vocabulary,
        measurement.measurement_datetime,
        measurement.measurement_type_concept_id,
        m_type.concept_name as measurement_type_concept_name,
        measurement.operator_concept_id,
        m_operator.concept_name as operator_concept_name,
        measurement.value_as_number,
        measurement.value_as_concept_id,
        m_value.concept_name as value_as_concept_name,
        measurement.unit_concept_id,
        m_unit.concept_name as unit_concept_name,
        measurement.range_low,
        measurement.range_high,
        measurement.visit_occurrence_id,
        m_visit.concept_name as visit_occurrence_concept_name,
        measurement.measurement_source_value,
        measurement.measurement_source_concept_id,
        m_source_concept.concept_name as source_concept_name,
        m_source_concept.concept_code as source_concept_code,
        m_source_concept.vocabulary_id as source_vocabulary,
        measurement.unit_source_value,
        measurement.value_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.measurement` measurement 
        WHERE
            (
                measurement_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (1619025, 3004249, 3012888, 3022318, 3025315, 3027018, 3031203, 3036277, 3038553, 36303797, 36306178, 36660257, 40759207, 40764999, 40765148)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                measurement.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (201820, 201826)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 ) 
                            AND age_at_event >= 18 ) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            person_id IN (SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                    WHERE
                                        concept_id IN (442793, 443732, 4193704)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) 
                                AND is_standard = 1 
                            UNION
                            DISTINCT SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                    WHERE
                                        concept_id IN (1567956, 44833365)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 0 
                                    AND is_selectable = 1) 
                                AND is_standard = 0 ) 
                            AND age_at_event >= 18 ) criteria ) 
                            AND cb_search_person.person_id IN (SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (43526465)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (44785829)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (45774751)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (793293)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (45774435)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (1583722)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (40170911)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (44506754)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (793143)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria ) 
                                AND cb_search_person.person_id IN (SELECT
                                    criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN (46236952, 3053283, 40764999, 40771922) 
                                    AND is_standard = 1 )) criteria ) )
                        )
                    ) measurement 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_standard_concept 
                        ON measurement.measurement_concept_id = m_standard_concept.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_type 
                        ON measurement.measurement_type_concept_id = m_type.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_operator 
                        ON measurement.operator_concept_id = m_operator.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_value 
                        ON measurement.value_as_concept_id = m_value.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_unit 
                        ON measurement.unit_concept_id = m_unit.concept_id 
                LEFT JOIn
                    `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                        ON measurement.visit_occurrence_id = v.visit_occurrence_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_visit 
                        ON v.visit_concept_id = m_visit.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` m_source_concept 
                        ON measurement.measurement_source_concept_id = m_source_concept.concept_id"""

dataset_33371088_measurement_df = pandas.read_gbq(
    dataset_33371088_measurement_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_33371088_measurement_df.head(5)

In [ ]:
import pandas
import os

# This query represents dataset "dataset_with_diseases" for domain "condition" and was generated for All of Us Registered Tier Dataset v7
dataset_33371088_condition_sql = """
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (134057, 316139, 320128, 440383)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            (concept_id IN(SELECT
                                DISTINCT c.concept_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                            JOIN
                                (SELECT
                                    CAST(cr.id as string) AS id       
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                WHERE
                                    concept_id IN (201820, 201826)       
                                    AND full_text LIKE '%_rank1]%'      ) a 
                                    ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                    OR c.path LIKE CONCAT('%.', a.id) 
                                    OR c.path LIKE CONCAT(a.id, '.%') 
                                    OR c.path = a.id) 
                            WHERE
                                is_standard = 1 
                                AND is_selectable = 1) 
                            AND is_standard = 1 ) 
                            AND age_at_event >= 18 ) criteria 
                    UNION
                    DISTINCT SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                        WHERE
                            person_id IN (SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                    WHERE
                                        concept_id IN (442793, 443732, 4193704)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) 
                                AND is_standard = 1 
                            UNION
                            DISTINCT SELECT
                                person_id 
                            FROM
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                            WHERE
                                concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                                    WHERE
                                        concept_id IN (1567956, 44833365)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 0 
                                    AND is_selectable = 1) 
                                AND is_standard = 0 ) 
                            AND age_at_event >= 18 ) criteria ) 
                            AND cb_search_person.person_id IN (SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (43526465)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (44785829)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (45774751)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (793293)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (45774435)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (1583722)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (40170911)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (44506754)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria 
                            UNION
                            DISTINCT SELECT
                                criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN(SELECT
                                        DISTINCT ca.descendant_id 
                                    FROM
                                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria_ancestor` ca 
                                    JOIN
                                        (SELECT
                                            DISTINCT c.concept_id       
                                        FROM
                                            `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c       
                                        JOIN
                                            (SELECT
                                                CAST(cr.id as string) AS id             
                                            FROM
                                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr             
                                            WHERE
                                                concept_id IN (793143)             
                                                AND full_text LIKE '%_rank1]%'       ) a 
                                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                                OR c.path LIKE CONCAT('%.', a.id) 
                                                OR c.path LIKE CONCAT(a.id, '.%') 
                                                OR c.path = a.id) 
                                        WHERE
                                            is_standard = 1 
                                            AND is_selectable = 1) b 
                                            ON (ca.ancestor_id = b.concept_id)) 
                                        AND is_standard = 1)) criteria ) 
                                AND cb_search_person.person_id IN (SELECT
                                    criteria.person_id 
                            FROM
                                (SELECT
                                    DISTINCT person_id, entry_date, concept_id 
                                FROM
                                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_all_events` 
                                WHERE
                                    (concept_id IN (46236952, 3053283, 40764999, 40771922) 
                                    AND is_standard = 1 )) criteria ) )
                        )
                    ) c_occurrence 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_standard_concept 
                        ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_type 
                        ON c_occurrence.condition_type_concept_id = c_type.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
                        ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` visit 
                        ON v.visit_concept_id = visit.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_source_concept 
                        ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
                LEFT JOIN
                    `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_status 
                        ON c_occurrence.condition_status_concept_id = c_status.concept_id"""

dataset_33371088_condition_df = pandas.read_gbq(
    dataset_33371088_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_33371088_condition_df.head(5)

In [ ]:
print(dataset_33371088_person_df.columns)
print(dataset_33371088_drug_df.columns)
print(dataset_33371088_measurement_df.columns)
print(dataset_33371088_person_df.shape)
print(dataset_33371088_drug_df.shape)
print(dataset_33371088_measurement_df.shape)

### The following cells deal with patient dataframe

In [ ]:
from datetime import datetime
dataset_person_concise = dataset_33371088_person_df[["person_id", "gender", "date_of_birth", "race", "ethnicity", "sex_at_birth"]].copy()
# Fixed reference date preserves the historical analysis; see limitations.
reference_date = pandas.Timestamp('2024-11-20', tz='UTC')
dataset_person_concise["age"] = (reference_date - dataset_person_concise['date_of_birth']).dt.days // 365
dataset_person_concise

### The following cells deal with drug dataframe

In [ ]:
dataset_33371088_drug_df["standard_concept_name"].unique()

# Change the drug name to universal format
dataset_drug_process = dataset_33371088_drug_df.copy()

# Using regex to replace the complex drug name to the simple name
dataset_drug_process["standard_concept_name"] = dataset_drug_process["standard_concept_name"].replace(to_replace=r'^.*canagliflozin.*$', value='canagliflozin', regex=True)
dataset_drug_process["standard_concept_name"] = dataset_drug_process["standard_concept_name"].replace(to_replace=r'^.*dapagliflozin.*$', value='dapagliflozin', regex=True)
dataset_drug_process["standard_concept_name"] = dataset_drug_process["standard_concept_name"].replace(to_replace=r'^.*empagliflozin.*$', value='empagliflozin', regex=True)
dataset_drug_process["standard_concept_name"] = dataset_drug_process["standard_concept_name"].replace(to_replace=r'^.*ertugliflozin.*$', value='ertugliflozin', regex=True)
dataset_drug_process["standard_concept_name"] = dataset_drug_process["standard_concept_name"].replace(to_replace=r'^.*dulaglutide.*$', value='dulaglutide', regex=True)
dataset_drug_process["standard_concept_name"] = dataset_drug_process["standard_concept_name"].replace(to_replace=r'^.*exenatide.*$', value='exenatide', regex=True)
dataset_drug_process["standard_concept_name"] = dataset_drug_process["standard_concept_name"].replace(to_replace=r'^.*liraglutide.*$', value='liraglutide', regex=True)
dataset_drug_process["standard_concept_name"] = dataset_drug_process["standard_concept_name"].replace(to_replace=r'^.*lixisenatide.*$', value='lixisenatide', regex=True)
dataset_drug_process["standard_concept_name"] = dataset_drug_process["standard_concept_name"].replace(to_replace=r'^.*semaglutide.*$', value='semaglutide', regex=True)

print(dataset_drug_process["standard_concept_name"].unique())
dataset_drug_process.head(10)

# Find the first exposure of the drug. 
patients = dataset_drug_process["person_id"].unique()

# Add drug type
drug_type_dict = {'ertugliflozin':"SGLT2i", 'exenatide':"GLP-1RA", 'liraglutide':"GLP-1RA", 'canagliflozin':"SGLT2i", 'empagliflozin':"SGLT2i",
 'dulaglutide':"GLP-1RA", 'dapagliflozin':"SGLT2i", 'semaglutide':"GLP-1RA", 'lixisenatide':"GLP-1RA",}

dataset_drug_process["initial_exposure"] = ""
dataset_drug_process["drug_type"] = dataset_drug_process["standard_concept_name"].map(drug_type_dict)
for patient in patients:
    patient_data_frame = dataset_drug_process.query("person_id == @patient")
    drug_exposure_list = patient_data_frame["drug_exposure_start_datetime"].to_list()
    earliest_exposure_time = min(drug_exposure_list)

    dataset_drug_process.loc[dataset_drug_process['person_id'] == patient, 'initial_exposure'] = earliest_exposure_time



dataset_drug_process["initial_exposure"] = pandas.to_datetime(dataset_drug_process["initial_exposure"])

In [ ]:
# Convert to weeks and do some calculations
dataset_drug_process["relative_drug_exposure_start_datetime_by_weeks"] = (dataset_drug_process["drug_exposure_start_datetime"] - dataset_drug_process["initial_exposure"]).dt.days / 7
dataset_drug_process["relative_drug_exposure_end_datetime_by_weeks"] = (dataset_drug_process["drug_exposure_end_datetime"] - dataset_drug_process["initial_exposure"]).dt.days / 7
dataset_drug_process["relative_exposure_week"] = dataset_drug_process["relative_drug_exposure_end_datetime_by_weeks"] - dataset_drug_process["relative_drug_exposure_start_datetime_by_weeks"]


In [ ]:
# remove unuseful columns
dataset_drug_concise = dataset_drug_process[["person_id", "standard_concept_name", "drug_type", "initial_exposure", "relative_drug_exposure_start_datetime_by_weeks", "relative_drug_exposure_end_datetime_by_weeks", "relative_exposure_week"]]
dataset_drug_concise = dataset_drug_concise.rename(columns={"standard_concept_name": "drug_name"})
dataset_drug_concise.to_csv("dataset_drug_concise.csv")
dataset_drug_concise

In [ ]:
patient_drug_type = dataset_drug_concise[["person_id", "drug_type"]].drop_duplicates()
type_1 = set(patient_drug_type.query("drug_type == 'SGLT2i'")["person_id"].to_list())
type_2 = set(patient_drug_type.query("drug_type == 'GLP-1RA'")["person_id"].to_list())


from matplotlib_venn import venn2, venn3
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 6))
venn2([type_1, type_2], set_labels=('SGLT2i', 'GLP-1RA'))
plt.title("Venn Diagram of SGLT2i and GLP-1RA")

# Filter the patients only have one medicine 
single_drug_patient_SGLT2i = type_1.difference(type_2)
single_drug_patient_GLP = type_2.difference(type_1)
len(single_drug_patient_SGLT2i), len(single_drug_patient_GLP)

single_drug_patient = single_drug_patient_SGLT2i.union(single_drug_patient_GLP)

### The following cells deal with measurement dataframe

In [ ]:
dataset_33371088_measurement_df["standard_concept_name"].value_counts()

In [ ]:
dataset_33371088_measurement_df["standard_concept_name"].unique()
dataset_measurement_process = dataset_33371088_measurement_df.copy()
# Unify Glomerular filteration rate. 

dataset_measurement_process["standard_concept_name"].unique()

kept_element = ['Systolic blood pressure', 'Body weight',
       'Body mass index (BMI) [Ratio]', 'Body height',
       'Diastolic blood pressure', 'Heart rate',
       'Glomerular filtration rate/1.73 sq M.predicted [Volume Rate/Area] in Serum, Plasma or Blood by Creatinine-based formula (CKD-EPI)',
       'PhenX - hip circumference protocol 020801',
       'Adult Waist Circumference Protocol', 'Heart rate rhythm',
       'Blood pressure panel']

dataset_measurement_process = dataset_measurement_process[dataset_measurement_process["standard_concept_name"].isin(kept_element)]
dataset_measurement_process["standard_concept_name"] = dataset_measurement_process["standard_concept_name"].replace(to_replace=r'^.*Glomerular.*$', value='GFR', regex=True)

dataset_measurement_process["standard_concept_name"].unique()

In [ ]:
# Get first drug exposure time
first_exposure = dataset_drug_concise[["person_id", "initial_exposure"]]
first_exposure = first_exposure.drop_duplicates()
first_exposure

# Merge it with measurement dataframe
pandas.set_option('display.max_columns', None)
dataset_measurement_process = dataset_measurement_process.merge(first_exposure, on = "person_id")
dataset_measurement_process["relative_measurement_datetime_by_weeks"] = (dataset_measurement_process["measurement_datetime"]-dataset_measurement_process["initial_exposure"]).dt.days / 7
dataset_measurement_concise = dataset_measurement_process[["person_id", "standard_concept_name", "measurement_datetime", "value_as_number", "unit_source_value", "initial_exposure", "relative_measurement_datetime_by_weeks"]]
dataset_measurement_concise = dataset_measurement_concise.rename(columns={"standard_concept_name": "measurement"})
dataset_measurement_concise

# Get the one close to initial exposure

In [ ]:
patient_with_GFR = dataset_measurement_concise[["person_id", "measurement"]]
patient_with_GFR = patient_with_GFR.query("measurement == 'GFR'")
patient_with_GFR = patient_with_GFR.drop_duplicates()
patient_with_GFR = patient_with_GFR["person_id"].to_list()

In [ ]:
# patient 
# time 0(first exposure), follow them for a certain period of time (36 months). Have the Glomerular measurement 
# censor: they drop or leave the study, or they didn't develop the events (no CDK measurement), 
# if you stop using the drug for more than 6 month, they got censor
# Event: they have the kidney disease (40% or more reduction in EGFR from the baseline(first measurement nearest to first exposure))

def check_whether_drop(weeks, threshold, end_month):
    sorted_number = sorted(weeks)
    sorted_number.append(end_month)
    
    for i in range(len(sorted_number)-1):
        if sorted_number[i + 1] - sorted_number[i] > threshold:
            return True
    return False

single_drug_patient_list = list(single_drug_patient)
measurement_list = ['Systolic blood pressure', 'Body weight',
       'Body mass index (BMI) [Ratio]', 'Body height',
       'Diastolic blood pressure', 'Heart rate',
       'PhenX - hip circumference protocol 020801',
       'Adult Waist Circumference Protocol', 'Heart rate rhythm',
       'Blood pressure panel', 'GFR']
n = 0

event_dataset = pandas.DataFrame(columns=['person_id', 'initial_GFR', 'final_GFR', 'Event20', 'Event40','final_measure_time', 'drug_type'])

for patient_id in single_drug_patient_list:
    # Check whether need to be filtered (drop or leave the study)
    three_year_to_weeks = 52*3
    six_month = 26
    patient_drug_dataset = dataset_drug_concise.query("person_id == @patient_id")
    patient_drug_start_time = patient_drug_dataset["relative_drug_exposure_start_datetime_by_weeks"].to_list()

    if patient_id not in patient_with_GFR:
        continue
#     if check_whether_drop(patient_drug_start_time, six_month, three_year_to_weeks):
#         continue
    n = n + 1
    drug_type = "SGLT2i"
    if patient_id in list(single_drug_patient_GLP):
        drug_type = "GLP-1RA"

    
    patient_measurement = dataset_measurement_concise.query("person_id == @patient_id & measurement == 'GFR'")
    measurement_time = patient_measurement["relative_measurement_datetime_by_weeks"].to_list()
    intial_measurement_time = min(measurement_time, key =abs)
    final_measurement_time = min(measurement_time, key=lambda x: abs(x - three_year_to_weeks))
#     patient_measurement = dataset_measurement_concise.query("person_id == @patient_id")
    initial_measurement_dataset = patient_measurement.query("relative_measurement_datetime_by_weeks == @intial_measurement_time")
    final_measurement_dataset = patient_measurement.query("relative_measurement_datetime_by_weeks == @final_measurement_time")
    
    initial_GFR = initial_measurement_dataset.query("measurement == 'GFR'")["value_as_number"].iloc[0]
    final_GFR = final_measurement_dataset.query("measurement == 'GFR'")["value_as_number"].iloc[0]
    Event20 = False

    if initial_GFR * 0.8 >= final_GFR:
        Event20 = True
        
    Event40 = False
    
    if initial_GFR * 0.6 >= final_GFR:
        Event40 = True
    event_dataset.loc[len(event_dataset)] = [patient_id, initial_GFR, final_GFR, Event20, Event40, final_measurement_time, drug_type]
    
event_dataset

    


    



In [ ]:
# event_counts = event_dataset['Event'].value_counts()

# event_counts.plot.bar()
# plt.show()

# drug_counts = event_dataset['drug_type'].value_counts()

# drug_counts.plot.bar()
# plt.show()

# # plot = event_dataset.plot.pie(y='Event', figsize=(5, 5))

In [ ]:
# event_counts

In [ ]:
final_dataset = event_dataset.merge(dataset_person_concise, on = "person_id", how = "left")

In [ ]:
final_dataset

### Process disease condition

In [ ]:
condition_dataset = dataset_33371088_condition_df.copy()

condition_dataset["standard_concept_name"] = condition_dataset["standard_concept_name"].replace(to_replace=r'^.*hypertens.*$', value='hypertension', regex=True)
condition_dataset["standard_concept_name"] = condition_dataset["standard_concept_name"].replace(to_replace=r'^.*heart failure.*$', value='heart failure', regex=True)
condition_dataset["standard_concept_name"] = condition_dataset["standard_concept_name"].replace(to_replace=r'^.*cardiovascular.*$', value='cardiovascular', regex=True)
condition_dataset["standard_concept_name"] = condition_dataset["standard_concept_name"].replace(to_replace=r'^.*depress.*$', value='depression', regex=True)
need = ["hypertension", "heart failure", "cardiovascular", "depression"]
condition_dataset = condition_dataset.query("standard_concept_name in @need")
condition_dataset.drop_duplicates()
condition_dataset




In [ ]:
condition_dataset_final = pandas.DataFrame(columns=['person_id', 'hypertension', 'heart_failure', 'cardiovascular', 'depression'])
id_list = final_dataset["person_id"].to_list()
for patient_id in id_list:
    person_id = patient_id
    query_data = condition_dataset.query("person_id == @patient_id")


    hypertension = ((query_data['standard_concept_name'] == 'hypertension') & (query_data['person_id'] == patient_id)).any()
    heart_failure = ((query_data['standard_concept_name'] == 'heart failure') & (query_data['person_id'] == patient_id)).any()
    cardiovascular = ((query_data['standard_concept_name'] == 'cardiovascular') & (query_data['person_id'] == patient_id)).any()
    depression = ((query_data['standard_concept_name'] == 'depression') & (query_data['person_id'] == patient_id)).any()
    
    condition_dataset_final.loc[len(condition_dataset_final)] = [person_id, hypertension, heart_failure, cardiovascular, depression]

condition_dataset_final

In [ ]:
final_dataset = final_dataset.merge(condition_dataset_final, on = "person_id", how = "left")
# drug_name = dataset_drug_concise[["person_id", "drug_name"]]
# drug_name = drug_name.drop_duplicates()
# final_dataset = final_dataset.merge(drug_name, on = "person_id", how = "left")
final_dataset = final_dataset.query("final_measure_time > 0")
final_dataset

In [ ]:
final_dataset["hypertension"].value_counts()

### Merge GFR with measurement 

In [ ]:
GFR = ['GFR']
GFR_measurement = dataset_measurement_concise.query("measurement in @GFR")
GFR_measurement["relative_measurement_datetime_by_weeks"] = (GFR_measurement["measurement_datetime"]-GFR_measurement["initial_exposure"]).dt.days / 7
mix_model_dataset = GFR_measurement.merge(final_dataset, on = "person_id", how = "right")
mix_model_dataset.head(20)

In [ ]:
# First, let's see what columns we have
print("All columns in the dataset:")
print(mix_model_dataset.columns.tolist())

# Basic information about the dataset
print("\nBasic dataset information:")
print(mix_model_dataset.info())

# Let's check how many unique patients we have
n_patients = mix_model_dataset['person_id'].nunique()
print(f"\nNumber of unique patients: {n_patients}")

# Distribution of drug types
print("\nDistribution of drug types:")
print(mix_model_dataset['drug_type'].value_counts())

# Basic statistics for key numeric variables
numeric_cols = ['value_as_number', 'initial_GFR', 'relative_measurement_datetime_by_weeks', 'age']
print("\nBasic statistics for numeric variables:")
print(mix_model_dataset[numeric_cols].describe())

# Check missing values
print("\nMissing values in each column:")
print(mix_model_dataset.isnull().sum())

# Distribution of demographic characteristics
print("\nDemographic distributions:")
for col in ['race', 'gender', 'ethnicity']:
    print(f"\n{col.upper()} distribution:")
    print(mix_model_dataset[col].value_counts())

In [ ]:
columns_to_keep = [
    'person_id',                                    # Random effect identifier
    'value_as_number',                             # GFR measurements (outcome)
    'measurement_datetime',                         # For potential time checks
    'relative_measurement_datetime_by_weeks',       # Time variable for model
    'initial_GFR',                                 # Baseline adjustment
    'final_GFR',                                   # For potential completeness checks
    'drug_type',                                   # Main predictor (SGLT2i vs GLP-1RA)
    'gender',                                      # Demographics
    'race',
    'ethnicity',
    'age',
    'hypertension',                               # Key comorbidities
    'cardiovascular',
    'depression'
]

clean_dataset = mix_model_dataset[columns_to_keep].copy()

# Show basic info about the cleaned dataset
print("Shape of cleaned dataset:", clean_dataset.shape)
print("\nMissing values in cleaned dataset:")
print(clean_dataset.isnull().sum())

In [ ]:
# 1. First, let's handle missing values
clean_dataset = clean_dataset.dropna(subset=[
    'value_as_number',  # Drop rows with missing GFR measurements
    'initial_GFR',      # Drop rows with missing baseline GFR
    'final_GFR'         # Drop rows with missing final GFR
])

# 2. Check the distribution of measurements per patient
measurements_per_patient = clean_dataset.groupby('person_id').size()
print("\nMeasurements per patient summary:")
print(measurements_per_patient.describe())

# 3. Let's look at the distribution of time points
print("\nTime points distribution:")
print(clean_dataset['relative_measurement_datetime_by_weeks'].describe())

# 4. Look at the balance between drug groups
print("\nDrug type distribution after cleaning:")
print(clean_dataset['drug_type'].value_counts())
print(clean_dataset['drug_type'].value_counts(normalize=True))

# 5. Create necessary binary/categorical variables
clean_dataset = clean_dataset.assign(
    # Convert drug_type to binary (1 for SGLT2i, 0 for GLP-1RA)
    drug_binary = lambda x: (x['drug_type'] == 'SGLT2i').astype(int),
    
    # Simplify gender (if needed)
    gender_binary = lambda x: (x['gender'] == 'Female').astype(int),
    
    # Create simplified race categories (adjust based on your needs)
    race_simplified = lambda x: x['race'].map({
        'White': 'White',
        'Black or African American': 'Black',
        'Asian': 'Asian',
        'More than one population': 'Other',
        'None Indicated': 'Other',
        'PMI: Skip': 'Other',
        'None of these': 'Other',
        'I prefer not to answer': 'Other',
        'Another single population': 'Other'
    })
)

print("\nShape after cleaning:", clean_dataset.shape)

# 6. Check the completeness of follow-up
time_range = clean_dataset.groupby('person_id')['relative_measurement_datetime_by_weeks'].agg(['min', 'max'])
print("\nFollow-up time summary (weeks):")
print(time_range.describe())

In [ ]:
# 1. Create GFR changes from baseline
clean_dataset['GFR_changes'] = clean_dataset['value_as_number'] - clean_dataset['initial_GFR']

# 2. Create treatment categories
clean_dataset['treatment_status'] = 'Untreated'  # default value
clean_dataset.loc[
    (clean_dataset['relative_measurement_datetime_by_weeks'] >= 0) & 
    (clean_dataset['drug_type'] == 'GLP-1RA'), 
    'treatment_status'] = 'GLP-1RA'
clean_dataset.loc[
    (clean_dataset['relative_measurement_datetime_by_weeks'] >= 0) & 
    (clean_dataset['drug_type'] == 'SGLT2i'), 
    'treatment_status'] = 'SGLT2i'

# 3. Adjust time for untreated period
clean_dataset['time_length'] = clean_dataset['relative_measurement_datetime_by_weeks']
clean_dataset.loc[clean_dataset['time_length'] < 0, 'time_length'] = 0

# 4. Create binary variables for diseases
disease_conditions = ['hypertension', 'cardiovascular', 'depression']

# 5. Prepare final dataset for mixed-effects model
model_dataset = clean_dataset.assign(
    sex_female = lambda x: (x['gender'] == 'Female').astype(int),
    # Standardize age
    age_scaled = lambda x: (x['age'] - x['age'].mean()) / x['age'].std()
)

# 6. Check the distribution of our new variables
print("Distribution of treatment status:")
print(model_dataset['treatment_status'].value_counts())
print("\nSummary of GFR changes:")
print(model_dataset['GFR_changes'].describe())
print("\nTime length summary:")
print(model_dataset['time_length'].describe())

# 7. Check number of measurements in each period per patient
measurements_by_treatment = model_dataset.groupby(['person_id', 'treatment_status']).size().unstack(fill_value=0)
print("\nMeasurements per patient in each period:")
print(measurements_by_treatment.describe())

# Optional: Create a visualization of GFR changes over time by treatment
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.boxplot(data=model_dataset, x='treatment_status', y='GFR_changes')
plt.title('GFR Changes by Treatment Status')
plt.show()

In [ ]:
# Display first few rows of the dataset with key columns
print("First few rows of the cleaned dataset:")
print(model_dataset[['person_id', 
                    'GFR_changes', 
                    'time_length', 
                    'treatment_status',
                    'value_as_number',
                    'initial_GFR',
                    'sex_female',
                    'age_scaled',
                    'hypertension',
                    'cardiovascular',
                    'depression']].head(10))

# Summary statistics for key numeric variables
print("\nSummary statistics for numeric variables:")
print(model_dataset[['GFR_changes', 
                    'time_length', 
                    'value_as_number', 
                    'initial_GFR',
                    'age_scaled']].describe())

# Check unique values in categorical variables
for col in ['treatment_status', 'hypertension', 'cardiovascular', 'depression']:
    print(f"\nDistribution of {col}:")
    print(model_dataset[col].value_counts(dropna=False))

# Check if we have any missing values in our key variables
print("\nMissing values in key variables:")
print(model_dataset[['GFR_changes', 
                    'time_length', 
                    'treatment_status',
                    'sex_female',
                    'age_scaled',
                    'hypertension',
                    'cardiovascular',
                    'depression']].isnull().sum())

In [ ]:
# 1. First look at distributions
print("gender distribution:")
print(clean_dataset['gender'].value_counts())
print("\nRace distribution:")
print(clean_dataset['race'].value_counts())

# 2. Create model dataset with all necessary variables
model_dataset = clean_dataset.assign(
    # Create sex_female from sex_at_birth
    sex_female = lambda x: (x['gender'] == 'Female').astype(int),
    
    # Standardize age
    age_scaled = lambda x: (x['age'] - x['age'].mean()) / x['age'].std(),
    
    # Standardize time_length
    time_length_scaled = lambda x: (x['time_length'] - x['time_length'].mean()) / x['time_length'].std(),
    
    # Race categories - using dummy variables
    race_white = lambda x: (x['race'] == 'White').astype(int),
    race_black = lambda x: (x['race'] == 'Black or African American').astype(int),
    race_asian = lambda x: (x['race'] == 'Asian').astype(int),
    race_other = lambda x: (~x['race'].isin(['White', 'Black or African American', 'Asian'])).astype(int),
    
    # Create drug dummy variables
    drug_GLP1RA = lambda x: (x['treatment_status'] == 'GLP-1RA').astype(int),
    drug_SGLT2i = lambda x: (x['treatment_status'] == 'SGLT2i').astype(int)
)

# 3. Show the updated dataset structure
print("\nFirst few rows of the updated dataset:")
print(model_dataset[['person_id', 
                    'GFR_changes', 
                    'time_length_scaled', 
                    'sex_female',
                    'race_white',
                    'race_black',
                    'race_asian',
                    'race_other',
                    'age_scaled',
                    'drug_GLP1RA',
                    'drug_SGLT2i',
                    'hypertension',
                    'cardiovascular',
                    'depression']].head(10))

# 4. Check distribution of categorical variables
print("\nDistribution of sex:")
print(f"Female: {model_dataset['sex_female'].mean()*100:.2f}%")

print("\nDistribution of race categories:")
for race_col in ['race_white', 'race_black', 'race_asian', 'race_other']:
    print(f"\n{race_col}:")
    print(model_dataset[race_col].value_counts())
    print(f"Percentage: {model_dataset[race_col].mean()*100:.2f}%")

# 5. Check for missing values
print("\nMissing values check:")
print(model_dataset[['GFR_changes', 
                    'time_length_scaled', 
                    'sex_female',
                    'race_white',
                    'race_black',
                    'race_asian',
                    'race_other',
                    'age_scaled',
                    'drug_GLP1RA',
                    'drug_SGLT2i',
                    'hypertension',
                    'cardiovascular',
                    'depression']].isnull().sum())

# 6. Summary statistics for standardized variables
print("\nSummary of standardized variables:")
print(model_dataset[['time_length_scaled', 'age_scaled']].describe())

In [ ]:
# 1. Create model dataset excluding the third gender category
model_dataset = clean_dataset[clean_dataset['gender'].isin(['Female', 'Male'])].assign(
    sex_female = lambda x: (x['gender'] == 'Female').astype(int),
    time_length_scaled = lambda x: (x['time_length'] - x['time_length'].mean()) / x['time_length'].std(),
    age_scaled = lambda x: (x['age'] - x['age'].mean()) / x['age'].std(),
    
    # Race categories
    race_white = lambda x: (x['race'] == 'White').astype(int),
    race_black = lambda x: (x['race'] == 'Black or African American').astype(int),
    race_asian = lambda x: (x['race'] == 'Asian').astype(int),
    race_other = lambda x: (~x['race'].isin(['White', 'Black or African American', 'Asian'])).astype(int),
    
    # Drug categories
    drug_GLP1RA = lambda x: (x['treatment_status'] == 'GLP-1RA').astype(int),
    drug_SGLT2i = lambda x: (x['treatment_status'] == 'SGLT2i').astype(int)
)

# 2. Check new sample size and distributions
print("Total sample size after excluding third gender category:", len(model_dataset))

print("\nGender distribution:")
print(model_dataset['sex_female'].value_counts())
print(f"Percentage female: {model_dataset['sex_female'].mean()*100:.2f}%")

print("\nRace distribution:")
for race_col in ['race_white', 'race_black', 'race_asian', 'race_other']:
    print(f"\n{race_col}:")
    print(model_dataset[race_col].value_counts())
    print(f"Percentage: {model_dataset[race_col].mean()*100:.2f}%")

print("\nDrug treatment distribution:")
for drug_col in ['drug_GLP1RA', 'drug_SGLT2i']:
    print(f"\n{drug_col}:")
    print(model_dataset[drug_col].value_counts())
    print(f"Percentage: {model_dataset[drug_col].mean()*100:.2f}%")

# 3. Show the updated dataset structure
print("\nFirst few rows of the final dataset:")
print(model_dataset[['person_id', 
                    'GFR_changes', 
                    'time_length_scaled', 
                    'sex_female',
                    'race_white',
                    'race_black',
                    'race_asian',
                    'race_other',
                    'age_scaled',
                    'drug_GLP1RA',
                    'drug_SGLT2i',
                    'hypertension',
                    'cardiovascular',
                    'depression']].head(10))

# 4. Check standardized variables
print("\nSummary of standardized variables:")
print(model_dataset[['time_length_scaled', 'age_scaled']].describe())

# 5. Verify no missing values
print("\nMissing values check:")
print(model_dataset[['GFR_changes', 
                    'time_length_scaled', 
                    'sex_female',
                    'race_white',
                    'race_black',
                    'race_asian',
                    'race_other',
                    'age_scaled',
                    'drug_GLP1RA',
                    'drug_SGLT2i',
                    'hypertension',
                    'cardiovascular',
                    'depression']].isnull().sum())

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Prepare the formula string for our mixed-effects model
model_formula = """GFR_changes ~ time_length_scaled + sex_female + 
                  race_black + race_asian + race_other + 
                  age_scaled + hypertension + cardiovascular + depression +
                  drug_GLP1RA + drug_SGLT2i"""

# Fit the mixed-effects model
mixed_model = smf.mixedlm(
    formula=model_formula,
    data=model_dataset,
    groups='person_id'
)

# Fit the model and store results
model_results = mixed_model.fit()

# Print summary
print("Mixed Effects Model Results:")
print(model_results.summary())

# Print confidence intervals
print("\nConfidence Intervals:")
print(model_results.conf_int())

# Calculate random effects variance
random_effects = model_results.random_effects
print("\nRandom Effects Summary:")
print(pd.concat([re.describe() for re in random_effects.values()], axis=1))

# Optional: Save predictions
model_dataset['predicted_values'] = model_results.predict(model_dataset)

# Calculate R-squared for the model
residuals = model_dataset['GFR_changes'] - model_dataset['predicted_values']
total_ss = ((model_dataset['GFR_changes'] - model_dataset['GFR_changes'].mean())**2).sum()
residual_ss = (residuals**2).sum()
r_squared = 1 - (residual_ss/total_ss)
print(f"\nR-squared: {r_squared:.3f}")

In [ ]:
import pandas as pd

# Save predictions
model_dataset['predicted_values'] = model_results.predict(model_dataset)

# Calculate R-squared
residuals = model_dataset['GFR_changes'] - model_dataset['predicted_values']
total_ss = ((model_dataset['GFR_changes'] - model_dataset['GFR_changes'].mean())**2).sum()
residual_ss = (residuals**2).sum()
r_squared = 1 - (residual_ss/total_ss)
print(f"\nR-squared: {r_squared:.3f}")

In [ ]:
# First, let's check current treatment variables
print("Current treatment distribution:")
print(model_dataset[['drug_GLP1RA', 'drug_SGLT2i']].value_counts())

# Create a single treatment categorical variable
model_dataset['treatment'] = 'Untreated'  # set default as Untreated
model_dataset.loc[model_dataset['drug_GLP1RA'] == 1, 'treatment'] = 'GLP1RA'
model_dataset.loc[model_dataset['drug_SGLT2i'] == 1, 'treatment'] = 'SGLT2i'

# Verify the new treatment variable
print("\nNew treatment distribution:")
print(model_dataset['treatment'].value_counts())

# Update model formula using categorical treatment with Untreated as reference
model_formula_updated = """GFR_changes ~ time_length_scaled + sex_female + 
                  race_black + race_asian + race_other + 
                  age_scaled + hypertension + cardiovascular + depression +
                  C(treatment, Treatment('Untreated'))"""

# Fit the updated model
mixed_model_updated = smf.mixedlm(
    formula=model_formula_updated,
    data=model_dataset,
    groups='person_id'
)

# Fit the model and store results
model_results_updated = mixed_model_updated.fit()

# Print summary
print("\nUpdated Mixed Effects Model Results:")
print(model_results_updated.summary())

# Calculate R-squared for the updated model
model_dataset['predicted_values'] = model_results_updated.predict(model_dataset)
residuals = model_dataset['GFR_changes'] - model_dataset['predicted_values']
total_ss = ((model_dataset['GFR_changes'] - model_dataset['GFR_changes'].mean())**2).sum()
residual_ss = (residuals**2).sum()
r_squared = 1 - (residual_ss/total_ss)
print(f"\nR-squared: {r_squared:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create time-series plot of GFR changes by treatment
plt.figure(figsize=(12, 6))
sns.lineplot(data=model_dataset, 
            x='time_length_scaled', 
            y='GFR_changes', 
            hue='treatment',
            ci=95)
plt.title('GFR Changes Over Time by Treatment')
plt.xlabel('Standardized Time')
plt.ylabel('GFR Change from Baseline')
plt.legend(title='Treatment')
plt.show()

In [ ]:
# Box plot of GFR changes by treatment
plt.figure(figsize=(10, 6))
sns.boxplot(data=model_dataset, 
           x='treatment', 
           y='GFR_changes')
plt.title('Distribution of GFR Changes by Treatment')
plt.xlabel('Treatment Group')
plt.ylabel('GFR Change from Baseline')
plt.show()

In [ ]:
# Random sample of patient trajectories
sample_patients = model_dataset['person_id'].sample(n=20, random_state=42).unique()
sample_data = model_dataset[model_dataset['person_id'].isin(sample_patients)]

plt.figure(figsize=(12, 6))
for patient in sample_patients:
    patient_data = sample_data[sample_data['person_id'] == patient]
    plt.plot(patient_data['time_length_scaled'], 
             patient_data['GFR_changes'], 
             alpha=0.5, 
             linewidth=1)
plt.title('Individual Patient GFR Trajectories (Sample of 20 Patients)')
plt.xlabel('Standardized Time')
plt.ylabel('GFR Change from Baseline')
plt.show()

In [ ]:
# Residual plots
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.scatterplot(data=model_dataset, 
                x='predicted_values', 
                y=residuals)
plt.axhline(y=0, color='r', linestyle='--')
plt.title('Residuals vs Fitted Values')
plt.xlabel('Fitted Values')
plt.ylabel('Residuals')

plt.subplot(1, 2, 2)
sns.histplot(residuals, kde=True)
plt.title('Distribution of Residuals')
plt.xlabel('Residuals')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create figure with multiple subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Baseline Characteristics by Treatment Group', fontsize=16)

# 1. Age Distribution
sns.boxplot(data=model_dataset, x='treatment', y='age', ax=axes[0,0])
axes[0,0].set_title('Age Distribution')
axes[0,0].set_ylabel('Age (years)')

# 2. Race Distribution
race_proportions = (model_dataset.groupby('treatment')
                   [['race_white', 'race_black', 'race_asian', 'race_other']]
                   .mean() * 100)
race_proportions.plot(kind='bar', ax=axes[0,1])
axes[0,1].set_title('Race Distribution')
axes[0,1].set_ylabel('Percentage')
axes[0,1].legend(['White', 'Black', 'Asian', 'Other'])

# 3. Initial GFR Distribution
sns.boxplot(data=model_dataset, x='treatment', y='initial_GFR', ax=axes[1,0])
axes[1,0].set_title('Initial GFR Distribution')
axes[1,0].set_ylabel('GFR (mL/min/1.73m²)')

# 4. Comorbidity Distribution
comorbidity_props = (model_dataset.groupby('treatment')
                    [['hypertension', 'cardiovascular', 'depression']]
                    .mean() * 100)
comorbidity_props.plot(kind='bar', ax=axes[1,1])
axes[1,1].set_title('Comorbidity Distribution')
axes[1,1].set_ylabel('Percentage')
axes[1,1].legend(['Hypertension', 'Cardiovascular', 'Depression'])

# Adjust layout
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Print summary statistics
print("\nSummary Statistics by Treatment Group:")
print(model_dataset.groupby('treatment').agg({
    'age': ['mean', 'std'],
    'initial_GFR': ['mean', 'std'],
    'hypertension': 'mean',
    'cardiovascular': 'mean',
    'depression': 'mean'
}).round(2))